# Entregável 1 — RecFair baseline (relatório)

> **Projeto:** RecFair — recomendação com contrato utilidade + justiça  
> **Arquitetura:** `baseline` · `prompt_version=v1`  
> **Runtime:** pacote `recfair/` · **Medição:** pacote `eval/`

Este notebook é **somente relatório**: especificação resumida e células que importam o sistema.

## Como reproduzir

```bash
cd recfair
python3.14 -m venv venv-recfair
source venv-recfair/bin/activate
cp .env.example .env   # GOOGLE_API_KEY
make install-dev
make kernel            # kernel Jupyter "Python (recfair)"
make chat ARCH=baseline
```

Golden-set: execute as células abaixo (`eval.runner.run_eval`).

## Estrutura herdada (import)

- Schema: `recfair.schemas.output.RecFairOutput`
- Golden-set: `data/golden/cases.json` + `eval.fingerprint.golden_revision()`
- Verify: `eval.verify.verify_case`
- Runner: `eval.runner.run_eval`

In [ ]:
import json
import os

from IPython.display import HTML, display

from eval.fingerprint import golden_revision, load_cases
from eval.report import build_results_table, render_comparison_report, render_metrics_panel, summarize_records
from eval.runner import run_eval
from recfair.config import apply_dotenv, ensure_google_api_key, model_version
from recfair.observability.run_record import git_sha

apply_dotenv()
print("chave:", ensure_google_api_key())
cases = load_cases()
print(len(cases), "casos | golden_revision =", golden_revision(cases))
print("modelo:", model_version(), "| git:", git_sha())

## Execução do golden-set (baseline)

Reexecuta os 30 casos via `run_eval`. Requer `GOOGLE_API_KEY` no ambiente.

In [ ]:
manifest = run_eval(arch="baseline", persist=True)
records = manifest["records"]
resumo = manifest["resumo"]
tabela = build_results_table(records)
display(HTML(render_comparison_report(tabela, "status_final")))
n_ok = int((tabela["status_final"] == "sucesso").sum())
n_err = int((tabela["status_final"] == "erro").sum())
n_err_star = int((tabela["status_final"] == "erro*").sum())
print(f"{len(records)} execuções · sucesso={n_ok} · erro={n_err} · erro*={n_err_star}")

## Métricas consolidadas

In [ ]:
restrict_ok = resumo["e1_rate_restrict_n"]
restrict_n = resumo["e1_rate_restrict_d"]
overall_ok = resumo["e1_rate_overall_n"]
overall_n = resumo["e1_rate_overall_d"]
display(HTML(render_metrics_panel(resumo, restrict_ok, restrict_n, overall_ok, overall_n)))
print(json.dumps(resumo, indent=2, ensure_ascii=False))
if "run_path" in manifest:
    print("salvo:", manifest["run_path"])

## Análise crítica (E1)

O baseline stuffing exercita interpretação de NL sobre tabelas CSV, mas falha em desempates (`cod_sku`), diversidade de marca e abstenção em consultas ambíguas (ex.: T09). Gaps G (preço, claim, estoque, PII, injection) são **falhas previstas** no E1.

**Hipótese E2:** text-to-SQL / tools para agregação determinística da janela 7d e preço; workflow LangGraph com `recursion_limit` explícito.

**Pergunta obrigatória (curso):** comparar `e1_rate_restrict` e `e1_rate_overall` entre `ARCH=baseline` e incrementos futuros, mesmo modelo, baseline reexecutado na mesma sessão.